# Polym Educational Trading Sandbox - EDA Template

Phase 2 - Task 79: Pre-built pandas logic for loading PostgreSQL cold-storage data
and performing visual backtesting analysis.

**Educational purpose only - paper trading simulation.**

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Database connection
from sqlalchemy import create_engine

# Configure display
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

## 1. Database Connection

Connect to the PostgreSQL cold storage database.

In [ ]:
# Database connection parameters
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'polym_sandbox',
    'user': 'polym',
    'password': 'polym_dev'
}

# Create connection string
conn_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

# Create SQLAlchemy engine
try:
    engine = create_engine(conn_string)
    print(f"Connected to PostgreSQL at {DB_CONFIG['host']}:{DB_CONFIG['port']}")
except Exception as e:
    print(f"Connection failed: {e}")
    engine = None

## 2. Load Tick Data

Load historical tick data for analysis.

In [ ]:
def load_ticks(symbol: str, hours: int = 24, engine=engine) -> pd.DataFrame:
    """
    Load tick data for a symbol from PostgreSQL.
    
    Args:
        symbol: Asset symbol (e.g., 'BTC', 'ETH')
        hours: Number of hours of data to load
        engine: SQLAlchemy engine
    
    Returns:
        DataFrame with tick data
    """
    query = f"""
    SELECT 
        timestamp,
        symbol,
        price,
        bid,
        ask,
        volume,
        source
    FROM ticks
    WHERE symbol = '{symbol}'
      AND timestamp > NOW() - INTERVAL '{hours} hours'
    ORDER BY timestamp ASC
    """
    
    if engine is None:
        print("No database connection. Generating sample data...")
        return generate_sample_ticks(symbol, hours)
    
    try:
        df = pd.read_sql(query, engine)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df.set_index('timestamp', inplace=True)
        print(f"Loaded {len(df):,} ticks for {symbol}")
        return df
    except Exception as e:
        print(f"Query failed: {e}")
        return generate_sample_ticks(symbol, hours)


def generate_sample_ticks(symbol: str, hours: int) -> pd.DataFrame:
    """Generate sample tick data for demonstration."""
    n_ticks = hours * 3600  # 1 tick per second
    
    base_prices = {'BTC': 65000, 'ETH': 3500, 'SOL': 150}
    base_price = base_prices.get(symbol, 100)
    
    timestamps = pd.date_range(
        end=datetime.now(),
        periods=n_ticks,
        freq='1s'
    )
    
    # Random walk for price
    returns = np.random.normal(0, 0.0001, n_ticks)
    prices = base_price * np.exp(np.cumsum(returns))
    
    # Bid/ask with spread
    spread = base_price * 0.0001
    bids = prices - spread / 2
    asks = prices + spread / 2
    
    df = pd.DataFrame({
        'symbol': symbol,
        'price': prices,
        'bid': bids,
        'ask': asks,
        'volume': np.random.uniform(0.1, 10, n_ticks),
        'source': 'simulated'
    }, index=timestamps)
    
    print(f"Generated {len(df):,} sample ticks for {symbol}")
    return df


# Load data for BTC and ETH
df_btc = load_ticks('BTC', hours=24)
df_eth = load_ticks('ETH', hours=24)

## 3. Calculate Features

Calculate VWAP, TWAP, returns, and other features.

In [ ]:
def calculate_features(df: pd.DataFrame, windows: list = [60, 300, 3600]) -> pd.DataFrame:
    """
    Calculate trading features from tick data.
    
    Args:
        df: DataFrame with price, bid, ask, volume columns
        windows: Rolling window sizes in seconds
    
    Returns:
        DataFrame with additional feature columns
    """
    df = df.copy()
    
    # Mid price and spread
    df['mid_price'] = (df['bid'] + df['ask']) / 2
    df['spread'] = df['ask'] - df['bid']
    df['spread_bps'] = (df['spread'] / df['mid_price']) * 10000
    
    # Returns
    df['return_1s'] = df['price'].pct_change()
    df['return_1m'] = df['price'].pct_change(60)
    df['return_5m'] = df['price'].pct_change(300)
    
    # Rolling calculations
    for window in windows:
        window_str = f'{window}s'
        suffix = f'_{window}s'
        
        # VWAP
        df[f'vwap{suffix}'] = (
            (df['price'] * df['volume']).rolling(window).sum() /
            df['volume'].rolling(window).sum()
        )
        
        # TWAP
        df[f'twap{suffix}'] = df['price'].rolling(window).mean()
        
        # Volatility
        df[f'volatility{suffix}'] = df['return_1s'].rolling(window).std() * np.sqrt(window)
    
    # Price deviation from VWAP
    df['price_vs_vwap'] = (df['price'] - df['vwap_60s']) / df['vwap_60s'] * 100
    
    return df


# Calculate features
df_btc = calculate_features(df_btc)
df_eth = calculate_features(df_eth)

print("\nBTC Features:")
df_btc.head()

## 4. Price Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# BTC Price
axes[0, 0].plot(df_btc.index, df_btc['price'], label='Price', alpha=0.8)
axes[0, 0].plot(df_btc.index, df_btc['vwap_300s'], label='VWAP (5m)', alpha=0.6)
axes[0, 0].set_title('BTC Price with VWAP')
axes[0, 0].legend()
axes[0, 0].set_ylabel('Price (USDT)')

# ETH Price
axes[0, 1].plot(df_eth.index, df_eth['price'], label='Price', alpha=0.8, color='purple')
axes[0, 1].plot(df_eth.index, df_eth['vwap_300s'], label='VWAP (5m)', alpha=0.6, color='orange')
axes[0, 1].set_title('ETH Price with VWAP')
axes[0, 1].legend()
axes[0, 1].set_ylabel('Price (USDT)')

# BTC Spread
axes[1, 0].plot(df_btc.index, df_btc['spread_bps'], alpha=0.7)
axes[1, 0].axhline(y=df_btc['spread_bps'].mean(), color='r', linestyle='--', label='Mean')
axes[1, 0].set_title('BTC Bid-Ask Spread')
axes[1, 0].set_ylabel('Spread (bps)')
axes[1, 0].legend()

# Volatility comparison
axes[1, 1].plot(df_btc.index, df_btc['volatility_300s'] * 100, label='BTC', alpha=0.7)
axes[1, 1].plot(df_eth.index, df_eth['volatility_300s'] * 100, label='ETH', alpha=0.7)
axes[1, 1].set_title('5-Minute Rolling Volatility')
axes[1, 1].set_ylabel('Volatility (%)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 5. Return Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Return distributions
for ax, (name, df) in zip(axes, [('BTC', df_btc), ('ETH', df_eth)]):
    returns = df['return_1m'].dropna() * 100
    
    ax.hist(returns, bins=100, density=True, alpha=0.7, label='Actual')
    
    # Fit normal distribution
    mu, std = returns.mean(), returns.std()
    x = np.linspace(returns.min(), returns.max(), 100)
    ax.plot(x, (1/(std * np.sqrt(2*np.pi))) * np.exp(-0.5*((x-mu)/std)**2),
            'r-', linewidth=2, label=f'Normal (μ={mu:.4f}, σ={std:.4f})')
    
    ax.set_title(f'{name} 1-Minute Return Distribution')
    ax.set_xlabel('Return (%)')
    ax.set_ylabel('Density')
    ax.legend()
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Statistics
print("\nReturn Statistics:")
for name, df in [('BTC', df_btc), ('ETH', df_eth)]:
    returns = df['return_1m'].dropna()
    print(f"\n{name}:")
    print(f"  Mean: {returns.mean()*100:.6f}%")
    print(f"  Std:  {returns.std()*100:.6f}%")
    print(f"  Skew: {returns.skew():.4f}")
    print(f"  Kurt: {returns.kurtosis():.4f}")

## 6. Correlation Analysis

In [ ]:
# Merge BTC and ETH on timestamp
df_merged = pd.merge(
    df_btc[['price', 'return_1m']].add_suffix('_btc'),
    df_eth[['price', 'return_1m']].add_suffix('_eth'),
    left_index=True,
    right_index=True,
    how='inner'
)

# Rolling correlation
df_merged['rolling_corr'] = df_merged['return_1m_btc'].rolling(300).corr(df_merged['return_1m_eth'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(
    df_merged['return_1m_btc'] * 100,
    df_merged['return_1m_eth'] * 100,
    alpha=0.3,
    s=1
)
axes[0].set_xlabel('BTC 1m Return (%)')
axes[0].set_ylabel('ETH 1m Return (%)')
axes[0].set_title(f'BTC vs ETH Returns (r = {df_merged["return_1m_btc"].corr(df_merged["return_1m_eth"]):.3f})')

# Rolling correlation
axes[1].plot(df_merged.index, df_merged['rolling_corr'])
axes[1].axhline(y=df_merged['rolling_corr'].mean(), color='r', linestyle='--')
axes[1].set_title('5-Minute Rolling Correlation')
axes[1].set_ylabel('Correlation')
axes[1].set_ylim(-1, 1)

plt.tight_layout()
plt.show()

## 7. Backtest Simulation

In [ ]:
def simple_momentum_backtest(
    df: pd.DataFrame,
    lookback: int = 60,
    threshold: float = 0.001,
    position_size: float = 100
) -> pd.DataFrame:
    """
    Simple momentum strategy backtest.
    
    Go long if recent return > threshold, short if < -threshold.
    """
    df = df.copy()
    
    # Signal: recent momentum
    df['momentum'] = df['price'].pct_change(lookback)
    
    # Position: 1 = long, -1 = short, 0 = flat
    df['position'] = 0
    df.loc[df['momentum'] > threshold, 'position'] = 1
    df.loc[df['momentum'] < -threshold, 'position'] = -1
    
    # Shift position to avoid lookahead bias
    df['position'] = df['position'].shift(1).fillna(0)
    
    # Strategy returns
    df['strategy_return'] = df['position'] * df['return_1s']
    
    # Cumulative PnL
    df['cumulative_pnl'] = (1 + df['strategy_return']).cumprod() * position_size - position_size
    df['buy_hold_pnl'] = (df['price'] / df['price'].iloc[0] - 1) * position_size
    
    return df


# Run backtest
bt_btc = simple_momentum_backtest(df_btc, lookback=60, threshold=0.0005)

# Plot results
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# PnL
axes[0].plot(bt_btc.index, bt_btc['cumulative_pnl'], label='Strategy')
axes[0].plot(bt_btc.index, bt_btc['buy_hold_pnl'], label='Buy & Hold', alpha=0.7)
axes[0].set_title('BTC Momentum Strategy Backtest')
axes[0].set_ylabel('PnL ($)')
axes[0].legend()
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Position
axes[1].fill_between(bt_btc.index, bt_btc['position'], alpha=0.5)
axes[1].set_ylabel('Position')
axes[1].set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

# Performance metrics
total_return = bt_btc['cumulative_pnl'].iloc[-1]
max_dd = (bt_btc['cumulative_pnl'].cummax() - bt_btc['cumulative_pnl']).max()
sharpe = bt_btc['strategy_return'].mean() / bt_btc['strategy_return'].std() * np.sqrt(86400)

print(f"\nBacktest Results:")
print(f"Total PnL: ${total_return:.2f}")
print(f"Max Drawdown: ${max_dd:.2f}")
print(f"Sharpe Ratio: {sharpe:.2f}")

## 8. Load Paper Trading Results

In [ ]:
def load_paper_trades(engine=engine, hours: int = 24) -> pd.DataFrame:
    """Load paper trading history from database."""
    query = f"""
    SELECT 
        timestamp,
        order_id,
        symbol,
        side,
        price,
        quantity,
        status,
        pnl
    FROM paper_trades
    WHERE timestamp > NOW() - INTERVAL '{hours} hours'
    ORDER BY timestamp ASC
    """
    
    if engine is None:
        # Generate sample trades
        n_trades = 50
        return pd.DataFrame({
            'timestamp': pd.date_range(end=datetime.now(), periods=n_trades, freq='30min'),
            'order_id': [f'order_{i}' for i in range(n_trades)],
            'symbol': np.random.choice(['BTC', 'ETH'], n_trades),
            'side': np.random.choice(['buy', 'sell'], n_trades),
            'price': np.random.uniform(0.95, 0.99, n_trades),
            'quantity': np.random.uniform(10, 100, n_trades),
            'status': 'filled',
            'pnl': np.random.normal(0.5, 2, n_trades)
        })
    
    try:
        return pd.read_sql(query, engine)
    except:
        return pd.DataFrame()


paper_trades = load_paper_trades()
if not paper_trades.empty:
    print(f"Loaded {len(paper_trades)} paper trades")
    print(f"\nTotal PnL: ${paper_trades['pnl'].sum():.2f}")
    print(f"Win Rate: {(paper_trades['pnl'] > 0).mean() * 100:.1f}%")
    paper_trades.head(10)

## 9. Summary Statistics

In [ ]:
print("=" * 60)
print("POLYM EDUCATIONAL SANDBOX - ANALYSIS SUMMARY")
print("=" * 60)

for name, df in [('BTC', df_btc), ('ETH', df_eth)]:
    print(f"\n{name}:")
    print(f"  Data points: {len(df):,}")
    print(f"  Time range: {df.index.min()} to {df.index.max()}")
    print(f"  Price range: ${df['price'].min():,.2f} - ${df['price'].max():,.2f}")
    print(f"  Avg spread: {df['spread_bps'].mean():.2f} bps")
    print(f"  Avg volatility (5m): {df['volatility_300s'].mean() * 100:.4f}%")

print("\n" + "=" * 60)
print("Educational simulation only - no real funds used.")
print("=" * 60)